# Логистическая регрессия: вероятность бинарного исхода

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Логистическая регрессия».

## Что делает метод

Логистическая регрессия моделирует вероятность бинарного исхода (болен или здоров, событие произошло или нет) как функцию предикторов. Метод связывает логарифм отношения шансов с линейной комбинацией предикторов; экспонента коэффициента — это отношение шансов, понятная мера влияния фактора.

Метод применим, когда отклик — бинарная метка, а предикторы заданы таблицей измерений. Типичные научные задачи: оценка риска по клиническим показателям, классификация образцов на два класса, анализ факторов, повышающих вероятность события.

В этом блокноте мы обучим логистическую регрессию, оценим качество классификации и разберём отношения шансов. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте весы: с одной стороны — аргументы «за» событие (повышенный холестерин, курение, возраст старше 60), с другой — «против» (низкое давление, активный образ жизни). Логистическая регрессия взвешивает каждый из этих аргументов и переводит итоговый перевес в вероятность от 0 до 1. Чем сильнее перевес в одну сторону, тем ближе вероятность к 1 (или к 0).

Принципиальное отличие от линейной регрессии: отклик здесь не непрерывное число, а принадлежность к одному из двух классов (болен / не болен, активен / неактивен, дефолт / нет). Линейная регрессия могла бы предсказать вероятность больше 1 или меньше 0, что бессмысленно. Логистическая регрессия решает эту проблему, применяя S-образную функцию (сигмоиду), которая сжимает любое значение в диапазон (0, 1).

**Ключевые термины, которые встретятся ниже:**
- **Признак** — измеримая характеристика объекта (диаметр клетки, уровень гормона).
- **Обучающая/тестовая выборка** — часть данных для обучения модели и часть для честной проверки её качества.
- **Метрика** — числовой показатель качества модели (доля верных ответов, AUC).
- **Отношение шансов** — во сколько раз меняются шансы исхода при изменении предиктора на единицу; значение > 1 означает рост шансов, < 1 — снижение.
- **ROC-кривая** — графический способ оценить, насколько хорошо модель разделяет классы при разных порогах принятия решения.
- **AUC** (площадь под ROC-кривой) — итоговая мера разделяющей способности; 0.5 — не лучше монетки, 1.0 — идеальная модель.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор по диагностике рака молочной железы (`breast_cancer` из `scikit-learn`): 569 наблюдений, 30 числовых признаков, описывающих характеристики клеточных ядер. Задача — отнести опухоль к доброкачественной или злокачественной.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target          # 1 - доброкачественная, 0 - злокачественная
feature_names = list(X.columns)
class_names = list(data.target_names)

print(f'Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}')
print(f'Классы: {class_names}')
X.head()

### Наглядный «ага»-эксперимент: S-образная кривая вероятности

Прежде чем работать с многомерными данными, посмотрим на суть метода на двух признаках. Возьмём два наиболее различающих класса признака и нарисуем поверхность вероятности — как модель оценивает шанс злокачественности в зависимости от значений двух измерений.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Выбираем два наиболее информативных признака для визуализации
feat_a, feat_b = 'mean radius', 'mean texture'
Xv = data.data[[feat_a, feat_b]].to_numpy()
yv = data.target.to_numpy()

sc = StandardScaler()
Xvs = sc.fit_transform(Xv)

clf2d = LogisticRegression(max_iter=1000).fit(Xvs, yv)

# Сетка для поверхности вероятности
h = 0.05
xx, yy = np.meshgrid(
    np.linspace(Xvs[:, 0].min() - 0.5, Xvs[:, 0].max() + 0.5, 200),
    np.linspace(Xvs[:, 1].min() - 0.5, Xvs[:, 1].max() + 0.5, 200))
Z = clf2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9.5, 6.2))
cf = ax.contourf(xx, yy, Z, levels=20,
                 cmap='RdBu_r', alpha=0.75, vmin=0, vmax=1)
plt.colorbar(cf, ax=ax, label='Вероятность доброкачественной опухоли')

for cls, color, label in zip(
        [0, 1],
        [VIZ['series'][2], VIZ['series'][0]],
        ['злокачественная', 'доброкачественная']):
    m = yv == cls
    ax.scatter(Xvs[m, 0], Xvs[m, 1],
               color=color, edgecolor='white', s=30, alpha=0.85,
               label=label, zorder=3)

# Граница решения (вероятность = 0.5)
ax.contour(xx, yy, Z, levels=[0.5],
           colors=[VIZ['series'][3]], linewidths=2,
           linestyles='--')

ax.set_xlabel(f'{feat_a} (стандартизованный)')
ax.set_ylabel(f'{feat_b} (стандартизованный)')
ax.set_title('Поверхность вероятности логистической регрессии')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

**Как читать этот график.** Цвет фона — вероятность доброкачественной опухоли согласно модели: синий (высокая вероятность) соответствует области, где ожидается доброкачественность; красный — злокачественность. Пунктирная линия — граница решения (вероятность = 0.5): левее неё модель классифицирует как злокачественную, правее — как доброкачественную. Точки — реальные наблюдения; видно, что граница в целом хорошо разделяет классы, но с некоторыми перекрытиями — это неизбежно при сложных реальных данных.

## 4. Применение метода

Разделим данные на обучающую и тестовую части и построим конвейер из двух шагов:
1. **Стандартизация** — приведение всех признаков к единому масштабу (среднее = 0, разброс = 1). Признаки набора `breast_cancer` измерены в разных единицах и имеют разный масштаб, поэтому стандартизация обязательна: без неё признак с большими числовыми значениями будет несправедливо доминировать.
2. **Логистическая регрессия** — собственно обучение модели.

`make_pipeline` — удобная обёртка, которая гарантирует, что стандартизация обучается только на тренировочных данных и применяется одинаково к тестовым.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000))
model.fit(X_train, y_train)
print('Модель обучена на', X_train.shape[0], 'наблюдениях.')

### Качество классификации

Следующая ячейка вычисляет метрики качества:

- **Доля верных ответов (accuracy)** — доля правильно классифицированных наблюдений. Простая, но не всегда достаточная метрика: если один класс редкий, высокая доля верных ответов достигается простым предсказанием «всегда большинство».
- **AUC ROC** — площадь под ROC-кривой. Показывает, насколько хорошо модель ранжирует наблюдения по риску, независимо от выбранного порога.
- **Точность (precision)** — из всех предсказанных как «положительные», какая доля действительно положительная. Важна, когда ложная тревога дорого обходится.
- **Полнота (recall)** — из всех истинно положительных, какую долю модель нашла. Важна, когда пропуск опасного случая недопустим.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f'Доля верных ответов: {accuracy_score(y_test, y_pred):.3f}')
print(f'AUC ROC: {roc_auc_score(y_test, y_proba):.3f}\n')
print(classification_report(y_test, y_pred, target_names=class_names))

### ROC-кривая и отношения шансов

**Зачем строить ROC-кривую?** Классификатор по умолчанию использует порог 0.5: если вероятность выше порога, объект относится к классу «1». Но в реальных задачах цена ошибок первого и второго рода различается. ROC-кривая показывает, как меняется компромисс между долей верно выявленных и долей ложных тревог при любом пороге от 0 до 1.

**Отношение шансов** (odds ratio) — экспонента коэффициента в логистической регрессии. Значение 2.5 означает: при увеличении стандартизованного предиктора на 1 шансы события растут в 2.5 раза. Значение 0.4 означает: шансы снижаются в 2.5 раза (0.4 = 1/2.5).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

clf = model.named_steps['logisticregression']
odds = pd.Series(np.exp(clf.coef_[0]), index=feature_names)
top = odds.reindex(odds.sub(1).abs().sort_values().index[::-1]).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0],
                                 color=VIZ['series'][0],
                                 name='логистическая регрессия')
axes[0].plot([0, 1], [0, 1], linestyle='--', color=VIZ['series'][3])
axes[0].set_xlabel('Доля ложных тревог')
axes[0].set_ylabel('Доля верно выявленных')
axes[0].set_title('ROC-кривая')

colors = [VIZ['series'][1] if v > 1 else VIZ['series'][2]
          for v in top.values]
axes[1].barh(top.index[::-1], top.values[::-1],
             color=colors[::-1])
axes[1].axvline(1.0, color=VIZ['series'][3], linestyle='--')
axes[1].set_xlabel('Отношение шансов')
axes[1].set_title('Наиболее влиятельные предикторы')

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый (ROC-кривая): ось X — доля ложных тревог (FPR), ось Y — доля верно выявленных положительных случаев (TPR). Диагональная пунктирная линия — случайный классификатор (монетка). Чем выше и левее кривая, тем лучше модель. Площадь AUC под кривой — итоговая мера: 1.0 — идеальная модель, 0.5 — случайная.

Правый (отношения шансов): столбики правее 1.0 (синие) — предикторы, повышающие шансы доброкачественности; левее 1.0 (оранжевые) — снижающие. Длина столбика отражает силу эффекта. Пунктирная вертикальная линия при значении 1.0 — точка нейтрального влияния.

## 5. Интерпретация результата

- **Доля верных ответов и AUC**: AUC около единицы означает почти идеальное разделение классов; 0.5 — отсутствие различающей способности.
- **Точность и полнота** по классам важны при несбалансированных данных: высокая полнота для класса риска означает, что модель редко пропускает опасные случаи.
- **Отношение шансов**: значение больше единицы — предиктор повышает шансы положительного исхода, меньше единицы — понижает. Отношение шансов вычислено для стандартизованных признаков, поэтому отражает изменение на одно стандартное отклонение.
- **ROC-кривая**: чем ближе кривая к левому верхнему углу, тем лучше модель разделяет классы при любом пороге.

Порог отнесения к классу (по умолчанию 0.5) можно сдвигать в зависимости от цены ошибок первого и второго рода.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу, где строки — наблюдения, столбцы — числовые предикторы, и отдельный столбец — бинарная метка (0 или 1).

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя файла и столбец с меткой.
3. Выполните ячейки разделов 4 и 5 — код переиспользуется без изменений.

## 7. Поэкспериментируйте сами

1. **Измените порог классификации.** По умолчанию граница решения — вероятность 0.5. Запустите код ниже, чтобы посмотреть, как меняются точность и полнота при разных порогах. В медицинских задачах часто выгодно снизить порог, чтобы не пропускать опасные случаи.

2. **Уберите стандартизацию.** В ячейке раздела 4 удалите `StandardScaler()` из конвейера и переобучите модель. Как изменятся AUC и качество? Это наглядно покажет, почему стандартизация важна.

3. **Попробуйте регуляризацию.** В `LogisticRegression` задайте параметр `C=0.01` (сильная регуляризация) и `C=100` (слабая). Как меняется число «значимых» отношений шансов?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_score, recall_score

# Смотрим, как меняются точность и полнота при разных порогах
thresholds = np.arange(0.1, 0.91, 0.05)
precisions, recalls = [], []
for thr in thresholds:
    y_thr = (y_proba >= thr).astype(int)
    precisions.append(precision_score(y_test, y_thr, zero_division=0))
    recalls.append(recall_score(y_test, y_thr, zero_division=0))

fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.plot(thresholds, precisions, color=VIZ['series'][0],
        marker='o', markersize=4, label='точность (precision)')
ax.plot(thresholds, recalls, color=VIZ['series'][1],
        marker='s', markersize=4, label='полнота (recall)')
ax.axvline(0.5, color=VIZ['series'][2], linestyle='--',
           label='порог по умолчанию = 0.5')
ax.set_xlabel('Порог вероятности')
ax.set_ylabel('Значение метрики')
ax.set_title('Зависимость точности и полноты от порога')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# target_column = 'класс'                  # бинарная метка 0/1
#
# y = df[target_column]
# X = df.drop(columns=[target_column]).select_dtypes('number')
# feature_names = list(X.columns)
# class_names = [str(c) for c in sorted(y.unique())]
#
# print(f'Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}')
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- Логистическая регрессия предсказывает вероятность бинарного исхода, применяя S-образную (сигмоидальную) функцию к линейной комбинации предикторов — вероятность всегда остаётся в диапазоне (0, 1).
- Коэффициенты интерпретируются через отношения шансов: значение > 1 означает, что предиктор повышает шансы события, < 1 — снижает.
- Качество оценивается несколькими метриками: для большинства задач важны AUC и баланс точности/полноты, а не только доля верных ответов.
- Порог классификации 0.5 — это умолчание, а не закон: для задач с разной ценой ошибок его нужно подбирать осознанно.
- Перед интерпретацией коэффициентов убедитесь, что признаки стандартизованы — иначе нельзя сравнивать величину коэффициентов между собой.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике отношений шансов у предиктора «mean radius» значение 0.35. Что это означает для интерпретации данного признака и как изменится ваш вывод, если предиктор не был стандартизован перед обучением?
2. Вы снижаете порог классификации с 0.5 до 0.3, чтобы реже пропускать злокачественные опухоли. Как при этом изменятся точность (precision) и полнота (recall), и почему этот компромисс зависит от задачи?
3. AUC ROC вашей модели равен 0.72, а у коллеги на тех же данных — 0.91. Прежде чем делать вывод о качестве, какие три вещи вы проверите в методологии его эксперимента?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Значение 0.35 означает: при увеличении стандартизованного предиктора на одно стандартное отклонение шансы доброкачественного исхода снижаются в 1/0.35 ≈ 2.86 раза. Если предиктор не стандартизован, отношение шансов отражает эффект изменения на одну единицу измерения — коэффициенты разных признаков становятся несопоставимы по величине, и их нельзя ранжировать напрямую.
2. При снижении порога полнота (recall) для класса «злокачественная» растёт: больше опасных случаев поймано. Одновременно точность (precision) падает: среди предсказанных как злокачественные становится больше фактически доброкачественных. Компромисс зависит от цены ошибок: в медицинской диагностике пропуск опасного случая обычно много опаснее лишней биопсии, поэтому снижение порога оправдано.
3. Стоит проверить: (а) нет ли утечки данных (признаки будущего или целевой переменной попали в обучение); (б) использовалась ли одна и та же тестовая выборка, или коллега многократно подбирал порог и отбирал признаки по тесту (оптимистичное смещение); (в) одинакова ли стратификация разбиения и не было ли повторных наблюдений одного субъекта в разных частях выборки.
</details>